# Phase 2A: Exploratory Statistical Analysis of Clean Normal Baseline

**Document:** Phase 2A Baseline Telemetry Profiling  
**Date:** September 28, 2026  
**Dataset Used:** `df_train_normal` (Run `21c851bc-384f-5b81-8747-6dcacdceff35` / `20260225_normal`)  

> **ANTI-LEAKAGE CERTIFICATION:**  
> This analysis strictly examines the uncompromised normal operational baseline. **ZERO** attack sessions, packet manipulations, or adversarial evaluation runs (`multi_*`) are loaded or referenced in this notebook.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data.duckdb_loader import NORMAL_DATASET_ID, DuckDBManager
from src.data.process_data import ProcessDataExtractor
from src.ml.baseline_analysis import BaselineAnalyzer

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Imports successful!')

## 1. Data Ingestion & Schema Verification

In [ ]:
analyzer = BaselineAnalyzer()
df = analyzer.load_data()

print(f'Normal Baseline Shape: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Metadata columns: {[c for c in ["dataset_id", "ts"]]}')
print(f'Control signals (C_*): {len(analyzer.control_cols)} signals')
print(f'Measurement signals (M_*): {len(analyzer.measurement_cols)} signals')
display(df.head(3))

## 2. Statistical Moments & Percentile Profiling

In [ ]:
stats_df = analyzer.calculate_summary_statistics()
display(stats_df.style.format({
    'mean': '{:.3f}',
    'std': '{:.3f}',
    'min': '{:.3f}',
    'p1': '{:.3f}',
    'median': '{:.3f}',
    'p99': '{:.3f}',
    'max': '{:.3f}',
    'iqr': '{:.3f}',
    'outlier_percentage': '{:.2f}%'
}))

## 3. Data Quality & Signal Behavior Screening

Identify constant signals, near-constant signals, and evaluate whether extreme values represent legitimate grid operations or sensor errors.

In [ ]:
anomalies = analyzer.detect_signal_anomalies()
print('=== Constant Signals (Zero Variance) ===')
for c in anomalies['constant_signals']:
    print(f"  {c['signal']}: fixed value = {c['constant_value']}")

print('\n=== Near-Constant Signals (Active <= 5% or <= 4 discrete values) ===')
for c in anomalies['near_constant_signals']:
    print(f"  {c['signal']}: {c['unique_values']} unique values, mode = {c['dominant_value']} ({c['dominant_frequency_pct']}% of samples)")

print('\n=== Highly Skewed Signals (|skew| > 2.0) ===')
for c in anomalies['highly_skewed_signals']:
    print(f"  {c['signal']}: skewness = {c['skewness']}")

## 4. Measurement Signal Correlation Matrix

In [ ]:
meas_corr = analyzer.compute_measurement_correlations()

# Find strong correlations (|r| >= 0.7)
strong = []
for i in range(len(analyzer.measurement_cols)):
    for j in range(i + 1, len(analyzer.measurement_cols)):
        c1 = analyzer.measurement_cols[i]
        c2 = analyzer.measurement_cols[j]
        val = meas_corr.loc[c1, c2]
        if abs(val) >= 0.7:
            strong.append({'Signal A': c1, 'Signal B': c2, 'Pearson r': round(float(val), 4)})

print(f'Total strongly coupled pairs (|r| >= 0.7): {len(strong)}')
display(pd.DataFrame(strong).sort_values(by='Pearson r', key=abs, ascending=False))

## 5. Control vs Measurement Coupling

In [ ]:
ctrl_meas_corr = analyzer.compute_control_measurement_correlations()
display(ctrl_meas_corr.style.format('{:.3f}'))

## 6. Temporal Continuity & Sampling Jitter

In [ ]:
temporal = analyzer.analyze_temporal_properties()
print(f"Total Samples: {temporal['total_samples']}")
print(f"Time Window: {temporal['start_time']} to {temporal['end_time']} ({temporal['duration_hours']} hours)")
print(f"Sampling Interval: mean = {temporal['sampling_interval_mean_sec']}s, median = {temporal['sampling_interval_median_sec']}s, std = {temporal['sampling_interval_std_sec']}s")
print(f"Sampling Min/Max: {temporal['sampling_interval_min_sec']}s / {temporal['sampling_interval_max_sec']}s")
print(f"Gaps > 2 seconds: {temporal['gaps_exceeding_2s_count']}")

## 7. Exploratory Visualizations

In [ ]:
plots = analyzer.generate_all_plots()
for name, path in plots.items():
    print(f'Generated: {name} -> {path.name}')

### 7.1 Representative Telemetry Dynamics
![Representative Time Series](../reports/figures/representative_time_series.png)

### 7.2 Measurement Correlation Matrix
![Correlation Heatmap](../reports/figures/correlation_heatmap.png)

### 7.3 Key Signal Distributions
![Signal Distributions](../reports/figures/signal_distributions.png)

### 7.4 Control vs Measurement Interactions
![Control vs Measurement](../reports/figures/control_vs_measurement.png)

### 7.5 Sampling Jitter Distribution
![Sampling Jitter](../reports/figures/sampling_interval_jitter.png)